<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">01 Data and Signals</h1>
<hr>


## **5. Entropy**


<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>

### 5.1 Introduction

The preceding notebooks introduced digital data, sampled time series, frequency-domain representations, and the bandwidth required to transmit time-varying signals. A sampled time series produces a sequence of data values, whether the samples are measured by a sensor, stored for later analysis, or transmitted through a communication channel. Before such data can be handled efficiently, it is useful to ask how much information the sequence actually contains and how much of it is predictable repetition.

This question matters throughout **data handling**. Every additional bit that must be stored, transmitted, or processed consumes memory, bandwidth, computation time, and energy. Compression before these operations can reduce that effort by removing statistical redundancy. It is important to be precise, however: lossless compression does not remove the information represented by entropy. Instead, entropy describes the theoretical lower limit toward which an efficient lossless representation can approach.

In information theory, entropy quantifies the average unpredictability of a data source. Data transmitted with, or obtained from, a time series has an entropy determined by the probabilities of its possible values or symbols. If a source always produces the same symbol, its uncertainty and entropy are zero. If it produces several symbols according to a probability distribution, its entropy reflects how difficult the next symbol is to predict. The formal definition is developed in Section 5.2.

A highly skewed distribution, where one outcome dominates, has low entropy because its output is fairly predictable. A uniform distribution, where all outcomes are equally likely, has maximum entropy. The following sections develop this intuition and use it to reason about compression, coding efficiency, and the effort required to handle data in resource-constrained mobile systems.


### 5.2 Entropy in Information Theory

In notebook [01.4, Converting Analog Signals to Digital Time Series](01.4_Converting_Analog_Signals_to_Digital_Time_Series.ipynb), a three-bit encoder represented each quantized sample by one of eight binary words: `000`, `001`, `010`, `011`, `100`, `101`, `110`, and `111`. Here, we treat each complete three-bit word as **one source symbol**, rather than treating its individual bits as separate symbols.

Every sample uses three bits in that fixed-length representation. However, this does not mean that every source has an entropy of three bits per symbol. Entropy measures the average uncertainty about which symbol occurs, based on the probabilities of the eight possible symbols. If one symbol occurs most of the time, the outcome is easier to predict. If all eight symbols are equally likely, the uncertainty is greatest.


#### 5.2.1 Event, Probability and Information

An event that occurs rarely, and therefore has a low probability, carries a greater degree of surprise and conveys more information than a routine, common event. In information theory, the information content (also called self-information) of an event $E$ with probability $p(E)$ is defined mathematically as

Information content of one event [bit]:
$$
    I(E) = \log_2 \left(\frac{1}{p(E)}\right) = -\log_2(p(E)).
$$

The negative logarithm ensures that events with smaller probabilities correspond to larger information values, while certain events ($p(E)=1$) yield zero information. The base 2 reflects the use of bit as the fundamental unit of information: a value of $I(E)=1$ bit means that observing the event resolves exactly one binary choice. In this way, the formula provides a direct link between probability, surprise, and the encoded length required for the outcome.

>For instance, consider the reliability of trains in Zurich. When a train arrives on time, this outcome is highly probable and therefore carries little new information. It confirms what passengers already expect, so the informational value is low. By contrast, if a train is delayed, this represents a rare and less probable event. Because it deviates from the expected pattern, it conveys substantially more information. The underlying principle is that the informational content of an event is inversely related to its probability: the less likely an outcome, the greater the amount of information it provides.



#### 5.2.2 Shannon Entropy Formula

The entropy $H(X)$ of a discrete random variable $X$ with outcomes $x_i$, each occurring with probability $p_i$, is defined as

$$
H(X) = \sum_i p_i I(x_i) = -\sum_{i} p_i \log_2 p_i .
$$

Each term weights a symbol's information content $I(x_i)=-\log_2 p_i$ by how often that symbol occurs. Adding the terms gives the average information per symbol. The probabilities sum to one; a symbol with zero probability contributes zero to the sum.

When $X$ represents one source symbol and $H(X)$ is calculated without using previous, following, or neighboring symbols as context, $H(X)$ is called the **symbol entropy**. It measures the average uncertainty of one symbol considered alone and is expressed in bit per source symbol.

> The formula is named after [Claude Shannon](https://en.wikipedia.org/wiki/Claude_Shannon), whose work established the foundations of information theory. We use the term "Entropy" for simplicity.

This expression represents the expected information content, or the average surprise, associated with the possible outcomes of the variable. In other words, Shannon entropy gives the lower bound on the average encoded length, measured in bit per source symbol, for samples drawn from the distribution of $X$. A distribution where one outcome dominates leads to low entropy because the outcome is highly predictable, whereas a uniform distribution, where all outcomes are equally likely, yields maximum entropy and therefore requires more information to specify the result.



#### 5.2.3 Examples with Three-Bit Symbols

We now apply the entropy formula to two probability distributions and compare the results with the decisions in each tree.

**A sequence of yes/no questions helps illustrate how much uncertainty a symbol carries.** Each answer is one binary decision. A decision tree specifies which question to ask next, depending on the previous answer.

First, suppose all eight three-bit symbols are equally likely, with probability $1/8$ each. We can identify a symbol in three questions by dividing the remaining possibilities into equal groups. For example, the path to `000` asks:

1. Is the symbol in the set {`000`, `001`, `010`, `011`}?
2. If yes, is it in the set {`000`, `001`}?
3. If yes, is it `000`?

A “no” answer follows a different branch, as shown below. Every path ends at one source symbol after exactly three decisions.

<p align="center"><img alt="Balanced decision tree for eight equally likely three-bit source symbols, with three yes/no decisions on every path" width="900" style="max-width:100%;height:auto" src="../../.admin/notebooks/picture/equal-probabilities.svg"><br><em>Equally likely three-bit source symbols. Each question asks whether the source symbol is in the indicated set.</em></p>

For each equally likely symbol, the information content is

$$
I(x)=-\log_2\left(\frac18\right)=3\ \text{bit}.
$$

Every symbol carries the same information content, so the average is also three bits per symbol. Applying the entropy formula gives

$$
H(X)=-8\left(\frac18\log_2\frac18\right)=3\ \text{bit per source symbol}.
$$

In this equal-probability example, the number of decisions matches the entropy exactly.

Now keep the same eight symbols, but change their probabilities:

| Source symbol | Probability $p(x)$ | Information content $I(x)=-\log_2 p(x)$ [bit] |
|:---:|:---:|:---:|
| `000` | $1/2$ | $1$ |
| `001` | $1/4$ | $2$ |
| `010` | $1/8$ | $3$ |
| Each of `011`, `100`, `101`, `110`, `111` | $1/40=0.025$ | $\log_2(40)\approx5.322$ |

The less likely a symbol is, the more information its occurrence carries. Information content depends on probability, not on the length of the original three-bit label. It can also be fractional: $5.322$ bits measures surprise, whereas an individual codeword or path through a binary tree uses a whole number of bits or decisions. Entropy is the probability-weighted average of these information-content values.

The balanced tree still identifies every symbol in three decisions. However, we can reduce the **average** number of decisions by asking about frequent symbols first:

1. Is the symbol `000`? A “yes” answer identifies it after just one decision. This happens half of the time.
2. If not, is it `001`? This symbol is identified after two decisions.
3. If not, is it `010`? This symbol is identified after three decisions.
4. If none of these applies, use further questions to distinguish the five less likely symbols.

<p align="center"><img alt="Unbalanced decision tree for three-bit source symbols: frequent symbols need one to three decisions, while rare symbols need five or six" width="1100" style="max-width:100%;height:auto" src="../../.admin/notebooks/picture/unequal-probabilities.svg"><br><em>Unequally likely three-bit source symbols. Each question asks whether the source symbol is in the indicated set.</em></p>

The labels at the ends of the branches are the original three-bit source symbols. The yes/no answers describe how we identify those symbols; they are not the individual bits of the original fixed-length representation.


In the second tree, `000`, `001`, and `010` require one, two, and three decisions, respectively. Three of the rare symbols require five decisions, and the other two require six. The average is therefore

$$
\overline{d}=\frac12(1)+\frac14(2)+\frac18(3)+\frac1{40}\bigl(3\cdot5+2\cdot6\bigr)=2.05
$$

decisions per source symbol. Some paths are longer than three decisions, but they occur rarely enough that the average falls below three.

**Entropy is a lower bound on the average number of binary decisions, not necessarily the exact average for a particular tree.** Using the probabilities of the eight symbols in the entropy formula gives

$$
H(X)=-\left(\frac12\log_2\frac12+\frac14\log_2\frac14+\frac18\log_2\frac18+5\cdot\frac1{40}\log_2\frac1{40}\right)
\approx2.04\ \text{bit per source symbol}.
$$

This is slightly below the tree's average of $2.05$ decisions. The original fixed-length representation still uses three bits per symbol. A probability-based encoding can use fewer bits on average by assigning shorter codewords to frequent symbols, as Section 5.3 illustrates.


### 5.3 From Entropy to Bits for Communication

We now turn the unequal-probability decision tree in Section 5.2.3 into a code for transmitting sensor samples. As in notebook 01.4, each quantized sample is represented by a three-bit word identifying one of eight levels. We use the same source symbols and probabilities as in the tree:

| Quantizer output (source symbol) | Probability | Uncompressed transmission | Compressed codeword |
|:--------------------------------:|:-----------:|:-------------------------:|:-------------------:|
| `000` | $1/2$ | `000` | `0` |
| `001` | $1/4$ | `001` | `10` |
| `010` | $1/8$ | `010` | `110` |
| `011` | $1/40$ | `011` | `11100` |
| `100` | $1/40$ | `100` | `11101` |
| `101` | $1/40$ | `101` | `11110` |
| `110` | $1/40$ | `110` | `111110` |
| `111` | $1/40$ | `111` | `111111` |

Each compressed codeword follows the path to its source symbol in the tree, writing `0` for “yes” and `1` for “no”. The continuation marker A adds no bit. No complete codeword is the beginning of another codeword, so a receiver that knows the tree can decode consecutive codewords without separators.

Without compression, each sample requires three bits, so the average encoded length is $L=3$ bit per source symbol. The probability-based code assigns shorter codewords to frequent symbols and longer codewords to rare symbols. The source symbol and its compressed codeword are different: for example, `000` is transmitted as `0`, while `111` is transmitted as `111111`. The sample values themselves are preserved.

With compression, the average encoded length $L$ equals the average number of decisions in the tree:

$$
L=\frac12(1)+\frac14(2)+\frac18(3)+\frac1{40}\bigl(3\cdot5+2\cdot6\bigr)
=2.05\ \text{bit per source symbol}.
$$

The entropy of this source is

$$
H(X)=-\left(\frac12\log_2\frac12+\frac14\log_2\frac14+\frac18\log_2\frac18+5\cdot\frac1{40}\log_2\frac1{40}\right)
\approx2.04\ \text{bit per source symbol}.
$$

This code comes close to the entropy limit but does not reach it exactly. For 1,000 symbols with exactly these proportions, fixed-length encoding uses 3,000 bits and the probability-based code uses 2,050 bits, a reduction of about 31.7%. This comparison counts only the encoded samples and assumes that the receiver already knows the code; it excludes headers and any description of the code itself. Under an independent-symbol model, encoding longer blocks of symbols can bring the average encoded length closer to the entropy limit.


#### 5.3.1 Comparing Sequences of Three-Bit Symbols

Each row below contains **16 source symbols**, with each symbol represented by three bits, as in the preceding examples. The eight possible symbols are `000` through `111`. Spaces separate symbols. Every row therefore uses $16\cdot3=48$ bits without compression, with average encoded length $L=3$ bit per source symbol.

We estimate the symbol probabilities by counting how often each complete three-bit symbol occurs:

$$
\widehat p_i=\frac{N_i}{16},\qquad
\widehat H=-\sum_{i=0}^{7}\widehat p_i\log_2\widehat p_i.
$$

Here, $N_i$ is the number of occurrences of symbol $i$. A symbol that does not appear contributes zero to the sum. The hats indicate estimates from the observed frequencies in this short sequence, rather than known probabilities of the source that generated it.

| Example | Sequence of 16 three-bit symbols | Estimated entropy [bit per source symbol] |
|:---|:---|---:|
| Only `000` | `000 000 000 000 000 000 000 000 000 000 000 000 000 000 000 000` | 0.000 |
| Only `111` | `111 111 111 111 111 111 111 111 111 111 111 111 111 111 111 111` | 0.000 |
| Eight `000`, then eight `111` | `000 000 000 000 000 000 000 000 111 111 111 111 111 111 111 111` | 1.000 |
| Alternating `000` and `111` | `000 111 000 111 000 111 000 111 000 111 000 111 000 111 000 111` | 1.000 |
| All eight symbols, repeated in order | `000 001 010 011 100 101 110 111 000 001 010 011 100 101 110 111` | 3.000 |
| All eight symbols, randomly shuffled | `010 000 100 001 010 101 001 111 000 101 011 110 100 110 111 011` | 3.000 |
| Unequal symbol frequencies, shuffled | `010 000 000 001 001 000 000 100 010 001 000 001 000 011 000 000` | 1.875 |

If every symbol is `000`, its observed probability is one and the estimated entropy is zero. The same applies when every symbol is `111`: what matters is the distribution of complete symbols, not how many individual zeros or ones they contain.

The two rows using only `000` and `111` have probabilities $1/2$ and $1/2$, so their estimated entropy is one bit per source symbol. In the rows containing all eight symbols twice, each probability is $2/16=1/8$, giving three bits per source symbol. The final row has nonzero probabilities $1/2$, $1/4$, $1/8$, $1/16$, and $1/16$, giving $\widehat H=1.875$ bit per source symbol.

**Why do an ordered sequence and a shuffled sequence have the same entropy when their symbol counts match?** This calculation considers how often each symbol occurs, but ignores its position and neighboring symbols. A repeating sequence can be easy to predict even when its symbol frequencies give maximum entropy. To describe predictability due to ordering, we need a model that accounts for relationships between symbols, such as conditional entropy or entropy rate. The next notebook on redundancy explores this distinction further, followed by the notebook on correlation.

These values describe the observed three-bit-symbol distribution. They do not establish that a sequence is random, nor do they give its exact compressed file size. In particular, multiplying $\widehat H$ by 16 does not account for patterns across positions or the information needed to describe a compression scheme. With only 16 observations, the estimated entropy can also differ from the entropy of the source that produced them.


---

### 5.4 Hands-On Exercise

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>



**For a coin, let $P(\mathrm{heads})=p$ and $P(\mathrm{tails})=1-p$. Calculate the entropy for $p=0.5$, $p=0.7$, and $p=0.99$. For each case, state both probabilities, show your calculation, and give the result in bit per coin toss, rounded to three decimal places. Explain why the entropy decreases as one outcome becomes more likely.**

(answer here)


**A fair six-sided die has $P(i)=1/6$ for each face $i=1,\ldots,6$. Calculate its entropy in bit per die roll. Show your calculation and round the result to three decimal places.**

(answer here)


**A weighted six-sided die has probabilities $[0.4,0.2,0.15,0.15,0.05,0.05]$ for faces 1 through 6, respectively. Check that the probabilities sum to one, then calculate its entropy in bit per die roll. Show your calculation and round the result to three decimal places.**

(answer here)


**Compare the fair and weighted dice described above. Which has lower entropy, and why? If you must guess one face before each roll, what is your best probability of a correct guess for each die? Relate these probabilities to your entropy comparison.**

(answer here)


**Use the three-bit source symbols, probabilities, and compressed codewords in Section 5.3. Calculate the information content $I(x)$ of `000` and `111`, the entropy $H(X)$ of the full eight-symbol distribution, and the average compressed codeword length $L$. Show how you weight the terms in $H(X)$ and $L$. Give units for each result. Explain why a symbol represented by three bits can carry more than three bits of information, and why $H(X)$ and $L$ are close but not identical.**

(answer here)


**In Section 5.3.1, compare the sequence containing all eight three-bit symbols in order, repeated twice, with the shuffled sequence containing the same symbols. Determine the observed probability of each symbol and calculate the estimated symbol entropy of each sequence. Why does the calculation give the same result even though the repeated order provides a prediction rule? State one kind of information about the sequence that this entropy calculation ignores.**

(answer here)


**Correct the marked logarithm expression in `compute_entropy` below so that it implements the Shannon entropy formula. Run the cell and use its three checks to assess your correction. In your written answer, explain why the function skips probabilities equal to zero.**

(answer here)


In [ ]:
import math

def compute_entropy(probabilities):
    if not math.isclose(sum(probabilities), 1.0, rel_tol=0.0, abs_tol=1e-6):
        print("WARNING in compute_entropy: The probabilities must sum to 1.")
        return "undefined"

    entropy = 0.0
    for p in probabilities:
        if p > 0:
            entropy -= p * math.log2(.6) ##### MODIFY HERE
    return entropy

implementation_is_correct = (
    abs(compute_entropy([0.25, 0.25, 0.25, 0.25]) - 2.0) < 1e-6 and
    abs(compute_entropy([0.5, 0.5]) - 1.0) < 1e-6 and
    abs(compute_entropy([1.0, 0.0]) - 0.0) < 1e-6
)

if implementation_is_correct:
    print("Well done! Your entropy function passes all checks.")
else:
    print("Not quite yet. Review which probability should be used inside the logarithm, then try again.")


**Complete the five probability lists below and use your corrected function to verify your hand calculations for the coins with heads probabilities 0.5, 0.7, and 0.99, the fair die, and the weighted die. Enter both coin outcomes in the order [heads, tails] and the die outcomes in face order 1 through 6. Run the cell after the function-definition cell. Compare every output with your hand calculation and explain any difference beyond rounding.**

(answer here)


In [ ]:
fair_coin_prob = [0.0, 0.0]   ### MODIFY HERE
print("Entropy of fair coin:", compute_entropy(fair_coin_prob), "bit per coin toss\n")

biased_coin_prob = [0.0 , 0.0] ### MODIFY HERE
print("Entropy of biased coin:", compute_entropy(biased_coin_prob), "bit per coin toss\n")

very_biased_coin_prob = [0.0, 0.0] ### MODIFY HERE
print("Entropy of very biased coin:", compute_entropy(very_biased_coin_prob), "bit per coin toss\n")

fair_die_prob = [1/6, 0.0, 0.0, 0.0, 0.0, 0.0]   ### MODIFY HERE
print("Entropy of fair die:", compute_entropy(fair_die_prob), "bit per die roll\n")

weighted_die_prob = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0] ### MODIFY HERE
print("Entropy of weighted die:", compute_entropy(weighted_die_prob), "bit per die roll\n")

**For the independent coin-toss and die-roll sources described above, what lower bound does each entropy value place on the average encoded length $L$ of a lossless code? Does a code that assigns a separate binary word to each outcome necessarily attain that bound exactly? Explain how the average length can be fractional even though every codeword contains a whole number of bits, and how encoding longer blocks can help approach the bound. Assume that the receiver knows the code and ignore headers.**

(answer here)


**OPTIONAL: Were any explanations, tasks, or visualizations unclear? Identify the section and describe what could be improved.**

(answer here)
